# pySEAFOM Spatial Resolution

**Repository**: SEAFOM-Fiber-Optic-Monitoring-Group/pySEAFOM

This notebook runs the spatial resolution analysis in the **pySEAFOM** library using synthetic or real DAS data.

## Overview

This notebook:

- Loads a 2D DAS matrix `.npy`
- Extracts a spatial section in the selected fibre interval
- Optionally converts phase (rad) → strain (µε) and applies a high-pass filter
- Runs the MSP-02 spatial resolution analysis at the reference tone
- Optionally generates:
  - **Spatiotemporal map**
  - **Spatial resolution profile**

## Key outputs (per run)

- **LL** and **LR** slope widths `[m]`
- **Spatial Resolution** `[m]`
- **Peak position** `[m]`
- **SNR** `[dB]`
- Optional figures written to `RESULTS_DIR`


## 1. Setup and Imports

Import `spatial_resolution` and the Python utilities.


In [ ]:
# Imports
import os
import numpy as np

try:
    from pySEAFOM import spatial_resolution
except ImportError:
    import spatial_resolution

## 2. Configure the Test Parameters

### Control
- `ANALYSIS_TYPE`: `"hilbert"`, `"thd"`, or `"both"`
- `SAVE_RESULTS`: if `True`, saves plots + CSV to `OUTPUT_DIR`
- `SHOW_PLOTS`: if `True`, displays figures

### Data extraction
DAS data is expected as a 2D `.npy` matrix with shape `(time, space)`.



In [ ]:
# ======================================================================
# CONTROL
# ======================================================================
SAVE_RESULTS = True                             # Saves figures in RESULTS_DIR
RESULTS_DIR = "results_spatial_resolution"
SHOW_PLOTS = True                               # Display figures (plt.show)
PLOT_METADATA_BOX = True                        # Add measurement-parameter box to the plot


# ======================================================================
# INPUT PARAMETERS (data extraction)
# ======================================================================
FOLDER_OR_FILE = "synthetic_sr_data.npy"        # Folder with .npy files OR a single .npy file
FS = 5000.0                                     # Sampling / interrogation rate [Hz]
DELTA_X_M = 1.0                                 # Spatial sampling interval [m]
MATRIX_LAYOUT = "space_time"                    # Input matrix layout

X1_M = 0.0                                      # Start position of the extracted section [m]
X2_M = 300.0                                    # End position of the extracted section [m]

TIME_START_S = 0.0                              # Start time of the extracted section [s]
DURATION_S = None                               # Duration of the extracted section [s]

DATA_IS_STRAIN = True                           # True if input data is already strain [µε]
GAUGE_LENGTH_M = 10.0                           # Gauge length [m]
HIGHPASS_HZ = None                              # Optional high-pass cutoff [Hz]


# ======================================================================
# ANALYSIS PARAMETERS
# ======================================================================
REF_FREQ_HZ = 100.0                             # Reference excitation frequency [Hz]
TARGET_POS_M = 150.0                            # Target / stretcher center position [m]
STRETCHER_LENGTH_M = 22.0                       # Stretcher physical length [m]

FFT_SIZE = 16384                                # FFT block size [samples]
SNR_THRESHOLD_DB = 20.0                         # Minimum SNR threshold [dB]

## 3. Load DAS Data and Extract the Spatial Section

This step loads a `.npy` matrix (or concatenates multiple `.npy` files from a folder),
then extracts the section bounded by `[X1_M, X2_M]` and the selected time interval.

Returned outputs:

- `time_s`: time vector (s)
- `x_vec_m`: spatial vector (m)
- `section_data`: extracted section with shape `(n_ssl, n_samples)`


In [ ]:
time_s, x_vec_m, section_data = spatial_resolution.load_spatial_resolution_data(
    folder_or_file=FOLDER_OR_FILE,
    fs=FS,
    delta_x_m=DELTA_X_M,
    x1_m=X1_M,
    x2_m=X2_M,
    time_start_s=TIME_START_S,
    duration=DURATION_S,
    matrix_layout=MATRIX_LAYOUT,
)

print(f"[INFO] Loaded: {os.path.abspath(FOLDER_OR_FILE)}")
print(f"[INFO] section_data.shape={section_data.shape} (space, time)")
print(f"[INFO] time=[{time_s[0]:.3f},{time_s[-1]:.3f}] s | n_time={time_s.size} | fs={FS}")
print(f"[INFO] dist=[{x_vec_m[0]:.1f},{x_vec_m[-1]:.1f}] m | n_space={x_vec_m.size} | dx={DELTA_X_M}")
print(f"[INFO] matrix_layout={'auto' if MATRIX_LAYOUT is None else MATRIX_LAYOUT} | data_is_strain={DATA_IS_STRAIN}")

## 4. Convert to Microstrain (and Optional High-Pass)

If `DATA_IS_STRAIN=False`, the trace is interpreted as **phase** (rad) and converted to strain (µε)
using the same physics used by `pySEAFOM.spatial_resolution.data_processing()`.


In [ ]:
section_data_ue = spatial_resolution.data_processing(
    trace=section_data,
    data_is_strain=DATA_IS_STRAIN,
    gauge_length=GAUGE_LENGTH_M,
    highpass_hz=HIGHPASS_HZ,
    fs=FS,
)

## 5. Spatial Resolution Analysis

Runs the spatial resolution calculation and, if enabled, generates:

- **Spatiotemporal map**
- **Spatial resolution profile**

Optional outputs are controlled by:

- `SHOW_PLOTS`
- `SAVE_RESULTS`
- `RESULTS_DIR`
- `PLOT_METADATA_BOX`


In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

result = spatial_resolution.calculate_spatial_resolution(
    # 1) required data
    section_data=section_data_ue,
    fs=FS,
    ref_freq_hz=REF_FREQ_HZ,
    target_pos_m=TARGET_POS_M,

    # 2) analysis parameters
    gauge_length=GAUGE_LENGTH_M,
    delta_x_m=DELTA_X_M,
    fft_size=FFT_SIZE,
    snr_threshold_db=SNR_THRESHOLD_DB,

    # 3) optional metadata / run context
    time_s=time_s,
    x_vec_m=x_vec_m,
    stretcher_length_m=STRETCHER_LENGTH_M,
    highpass_hz=HIGHPASS_HZ,

    # 4) I/O options
    show_plot=SHOW_PLOTS,
    save_results=SAVE_RESULTS,
    results_dir=RESULTS_DIR,
    plot_metadata_box=PLOT_METADATA_BOX,
    plot_spatiotemporal_map=True,
    plot_profile=True,
    print_report=True,
)

print("\n=== Spatial Resolution Summary ===")
print(f"Peak position        : {result['peak_pos_m']:.3f} m")
print(f"LL                   : {result['LL_m']:.3f} m")
print(f"LR                   : {result['LR_m']:.3f} m")
print(f"Spatial Resolution   : {result['spatial_resolution_m']:.3f} m")
print(f"SNR                  : {result['snr_db']:.2f} dB")

## 6. Minimal Calls (Examples)

Use these if you only want a quick run with default settings.


In [ ]:
# Minimal call (quick run)
# spatial_resolution.calculate_spatial_resolution(
#     section_data=section_data_ue,
#     fs=FS,
#     ref_freq_hz=REF_FREQ_HZ,
#     target_pos_m=TARGET_POS_M,
# )

# Minimal call + metadata box
# spatial_resolution.calculate_spatial_resolution(
#     section_data=section_data_ue,
#     fs=FS,
#     ref_freq_hz=REF_FREQ_HZ,
#     target_pos_m=TARGET_POS_M,
#     plot_metadata_box=PLOT_METADATA_BOX,
# )

## 7. Output Artifacts

If `SAVE_RESULTS=True`, the following are written to `RESULTS_DIR`:

- `spatiotemporal_map.png`
- `spatial_resolution_profile.png`
- `spatial_resolution_summary.csv`

Reported metrics include:
- LL slope width `[m]`
- LR slope width `[m]`
- Spatial Resolution `[m]`
- Peak position `[m]`